# Phase 1 Dataset Distribution Audit

This notebook verifies the distribution statistics reported in `reports/phase1_distribution_audit.md`.

Input dataset: `data/final/vismishds_phase1_final.csv`

In [33]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'final' / 'vismishds_phase1_final.csv'

df = pd.read_csv(DATA_PATH)
df.head()

,sample_id,content,label,has_url,has_phone_number,sender_type,category,obfuscation_level,data_origin,source_dataset,source_file,source_row_id
0,phase1_00001,Hương đang làm gì vậy? Dạo này công việc thế n...,0,0,0,personal_number,Cá nhân & OTP,NONE,real,real_label_0,data/ground_truth/dataset_label_0_with_categor...,0
1,phase1_00002,"Linh ơi anh Tuấn nè, hôm nay vô tình gặp chị M...",0,0,0,personal_number,Cá nhân & OTP,NONE,real,real_label_0,data/ground_truth/dataset_label_0_with_categor...,1
2,phase1_00003,"Thảo dạo này khoẻ không, công việc ổn chứ ?",0,0,0,personal_number,Cá nhân & OTP,NONE,real,real_label_0,data/ground_truth/dataset_label_0_with_categor...,2
3,phase1_00004,Chào Phương! Bạn và gia đình dạo này khoẻ chứ?,0,0,0,personal_number,Cá nhân & OTP,NONE,real,real_label_0,data/ground_truth/dataset_label_0_with_categor...,3
4,phase1_00005,Chào Hùng. Lâu rồi không gặp. dạo này công việ...,0,0,0,personal_number,Cá nhân & OTP,NONE,real,real_label_0,data/ground_truth/dataset_label_0_with_categor...,4


## Basic Dataset Information

In [34]:
print('Dataset path:', DATA_PATH)
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

display(df.dtypes.to_frame('dtype'))

Dataset path: c:\KLTN\KLTN\data\final\vismishds_phase1_final.csv
Shape: (10562, 12)

Columns:
['sample_id', 'content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'source_dataset', 'source_file', 'source_row_id']


,dtype
sample_id,object
content,object
label,int64
has_url,int64
has_phone_number,int64
sender_type,object
category,object
obfuscation_level,object
data_origin,object
source_dataset,object


## Helper Functions

In [35]:
def count_table(data: pd.DataFrame, cols, total=None) -> pd.DataFrame:
    if isinstance(cols, str):
        cols = [cols]
    if total is None:
        total = len(data)
    out = (
        data.groupby(cols, dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values(cols)
        .reset_index(drop=True)
    )
    out['ratio_pct'] = (out['count'] / total * 100).round(2)
    return out


def count_table_desc(data: pd.DataFrame, cols, total=None) -> pd.DataFrame:
    out = count_table(data, cols, total=total)
    return out.sort_values('count', ascending=False).reset_index(drop=True)


total = len(df)
total

10562

## 1. Label Distribution

In [36]:
label_dist = count_table(df, 'label', total)
display(label_dist)

,label,count,ratio_pct
0,0,5320,50.37
1,1,5242,49.63


## 2. Data Origin Distribution

In [37]:
origin_dist = count_table(df, 'data_origin', total)
display(origin_dist)

,data_origin,count,ratio_pct
0,real,2567,24.3
1,synthetic,7995,75.7


## 3. Label by Data Origin

This table is the most important distribution check because label 1 is mostly synthetic.

In [38]:
label_origin = count_table(df, ['label', 'data_origin'], total)
display(label_origin)

label_origin_within_label = label_origin.copy()
label_totals = df.groupby('label').size().rename('label_total')
label_origin_within_label = label_origin_within_label.merge(label_totals, on='label')
label_origin_within_label['within_label_pct'] = (
    label_origin_within_label['count'] / label_origin_within_label['label_total'] * 100
).round(2)
display(label_origin_within_label)

,label,data_origin,count,ratio_pct
0,0,real,2321,21.98
1,0,synthetic,2999,28.39
2,1,real,246,2.33
3,1,synthetic,4996,47.30


,label,data_origin,count,ratio_pct,label_total,within_label_pct
0,0,real,2321,21.98,5320,43.63
1,0,synthetic,2999,28.39,5320,56.37
2,1,real,246,2.33,5242,4.69
3,1,synthetic,4996,47.30,5242,95.31


## 4. Category Distribution

In [39]:
category_dist = count_table_desc(df, 'category', total)
display(category_dist)

,category,count,ratio_pct
0,Viễn thông,1360,12.88
1,Nội dung nhạy cảm,732,6.93
2,Cờ bạc / Betting,721,6.83
3,Crypto / Đầu tư giả,719,6.81
4,Tuyển dụng giả,703,6.66
5,BHXH / Trợ cấp,678,6.42
6,Ngân hàng thật,576,5.45
7,Dịch vụ công giả,565,5.35
8,Khác,553,5.24
9,Giả mạo ngân hàng,519,4.91


## 5. Category by Label

In [41]:
CATEGORY_MAP = {
    'Cờ bạc / Betting': 'Cờ bạc / Betting',
    'Cờ bạc/Betting': 'Cờ bạc / Betting',

    'BHXH / Trợ cấp': 'BHXH / Trợ cấp giả',
    'BHXH/Trợ cấp giả': 'BHXH / Trợ cấp giả',

    'Đòi nợ / Đe dọa': 'Đòi nợ / Đe dọa',
    'Đòi nợ/Đe doạ': 'Đòi nợ / Đe dọa',

    'Crypto / Đầu tư giả': 'Crypto / Đầu tư giả',
    'Đầu tư/Crypto giả': 'Crypto / Đầu tư giả',

    'Dịch vụ y tế': 'Dịch vụ y tế',
    'Y tế': 'Dịch vụ y tế',

    'Tin nhắn cá nhân và OTP': 'Cá nhân & OTP',
    'Cá nhân & OTP': 'Cá nhân & OTP',
}

df['category_normalized'] = df['category'].replace(CATEGORY_MAP)

In [42]:
display(
    df.groupby(['label', 'category_normalized'])
      .size()
      .reset_index(name='count')
      .sort_values(['label', 'category_normalized'])
)

,label,category_normalized,count
0,0,Cá nhân & OTP,801
1,0,Dịch vụ công thật,470
2,0,Dịch vụ y tế,302
3,0,Khác,507
4,0,Ngân hàng thật,576
5,0,Quảng cáo hợp lệ,486
6,0,Thương mại điện tử,400
7,0,Viễn thông,1360
8,0,Vận chuyển,418
9,1,BHXH / Trợ cấp giả,693


## 6. URL and Phone Number Distribution

In [43]:
has_url_dist = count_table(df, 'has_url', total)
has_phone_dist = count_table(df, 'has_phone_number', total)
label_url_phone = count_table(df, ['label', 'has_url', 'has_phone_number'], total=None).drop(columns=['ratio_pct'])

display(has_url_dist)
display(has_phone_dist)
display(label_url_phone)

,has_url,count,ratio_pct
0,0,4956,46.92
1,1,5606,53.08


,has_phone_number,count,ratio_pct
0,0,7515,71.15
1,1,3047,28.85


,label,has_url,has_phone_number,count
0,0,0,0,2739
1,0,0,1,913
2,0,1,0,987
3,0,1,1,681
4,1,0,0,634
5,1,0,1,670
6,1,1,0,3155
7,1,1,1,783


## 7. URL and Phone Number by Label and Origin

This extra table helps separate real and synthetic behavior.

In [44]:
label_origin_url_phone = count_table(
    df,
    ['label', 'data_origin', 'has_url', 'has_phone_number'],
    total=None,
).drop(columns=['ratio_pct'])
display(label_origin_url_phone)

,label,data_origin,has_url,has_phone_number,count
0,0,real,0,0,799
1,0,real,0,1,391
2,0,real,1,0,566
3,0,real,1,1,565
4,0,synthetic,0,0,1940
5,0,synthetic,0,1,522
6,0,synthetic,1,0,421
7,0,synthetic,1,1,116
8,1,real,0,0,35
9,1,real,0,1,30


## 8. Message Length Distribution

In [45]:
df_len = df.copy()
df_len['message_length'] = df_len['content'].fillna('').astype(str).str.len()

length_by_label_origin = (
    df_len.groupby(['label', 'data_origin'])['message_length']
    .agg(
        count='count',
        avg='mean',
        min='min',
        p50='median',
        p90=lambda s: s.quantile(0.90),
        max='max',
    )
    .reset_index()
)

length_by_label_origin['avg'] = length_by_label_origin['avg'].round(1)
length_by_label_origin['p90'] = length_by_label_origin['p90'].round(0).astype(int)
length_by_label_origin['p50'] = length_by_label_origin['p50'].round(0).astype(int)

display(length_by_label_origin)

,label,data_origin,count,avg,min,p50,p90,max
0,0,real,2321,228.9,2,227,362,916
1,0,synthetic,2999,112.3,23,110,162,219
2,1,real,246,194.1,49,156,352,920
3,1,synthetic,4996,117.5,12,110,182,298


## 9. Summary Checks

In [47]:
label1 = df[df['label'] == 1]
label1_real = label1[label1['data_origin'] == 'real']
label1_synthetic = label1[label1['data_origin'] == 'synthetic']

summary = pd.DataFrame([
    {'metric': 'total_samples', 'value': len(df)},
    {'metric': 'label_0_samples', 'value': int((df['label'] == 0).sum())},
    {'metric': 'label_1_samples', 'value': int((df['label'] == 1).sum())},
    {'metric': 'real_samples', 'value': int((df['data_origin'] == 'real').sum())},
    {'metric': 'synthetic_samples', 'value': int((df['data_origin'] == 'synthetic').sum())},
    {'metric': 'label_1_real_samples', 'value': len(label1_real)},
    {'metric': 'label_1_synthetic_samples', 'value': len(label1_synthetic)},
    {'metric': 'label_1_real_ratio_pct', 'value': round(len(label1_real) / len(label1) * 100, 2)},
    {'metric': 'label_1_synthetic_ratio_pct', 'value': round(len(label1_synthetic) / len(label1) * 100, 2)},
])

display(summary)

,metric,value
0,total_samples,10562.00
1,label_0_samples,5320.00
2,label_1_samples,5242.00
3,real_samples,2567.00
4,synthetic_samples,7995.00
5,label_1_real_samples,246.00
6,label_1_synthetic_samples,4996.00
7,label_1_real_ratio_pct,4.69
8,label_1_synthetic_ratio_pct,95.31


## Optional: Export Audit Tables

Run the cell below if you want CSV versions of the audit tables under `reports/phase1_distribution_tables/`.

In [48]:
EXPORT_TABLES = False

if EXPORT_TABLES:
    out_dir = ROOT / 'reports' / 'phase1_distribution_tables'
    out_dir.mkdir(parents=True, exist_ok=True)

    tables = {
        'label_distribution': label_dist,
        'origin_distribution': origin_dist,
        'label_origin_distribution': label_origin,
        'label_origin_within_label': label_origin_within_label,
        'category_distribution': category_dist,
        'label_category_distribution': label_category,
        'has_url_distribution': has_url_dist,
        'has_phone_distribution': has_phone_dist,
        'label_url_phone_distribution': label_url_phone,
        'label_origin_url_phone_distribution': label_origin_url_phone,
        'length_by_label_origin': length_by_label_origin,
        'source_dataset_distribution': source_dist,
        'label_source_dataset_distribution': label_source,
        'summary_checks': summary,
    }

    for name, table in tables.items():
        table.to_csv(out_dir / f'{name}.csv', index=False, encoding='utf-8-sig')

    print('Exported tables to:', out_dir)